In [59]:
import pandas as pd
import numpy as np
import lightgbm as lgb


train_model = pd.read_csv("/Users/JulioJerez/Desktop/Docs. Personales/Repositorio/PROYECTOS/store-sales-forecasting/data/processed/train_model.csv", parse_dates=["date"])
test_fe     = pd.read_csv("/Users/JulioJerez/Desktop/Docs. Personales/Repositorio/PROYECTOS/store-sales-forecasting/data/processed/test_fe.csv", parse_dates=["date"])

print(train_model.shape, test_fe.shape)

(2901096, 57) (28512, 56)


In [ ]:
cutoff = train_model["date"].max() - pd.Timedelta(days=28)

train_tr = train_model[train_model["date"] <= cutoff].copy()
train_va = train_model[train_model["date"] > cutoff].copy()

print(train_tr["date"].min(), train_tr["date"].max())
print(train_va["date"].min(), train_va["date"].max())

2013-02-26 00:00:00 2017-07-18 00:00:00
2017-07-19 00:00:00 2017-08-15 00:00:00


In [61]:
def rmsle(y_true, y_pred):
    y_pred = np.maximum(y_pred, 0)  # por seguridad
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true))**2))

In [62]:
simple_cols = [
    "sales_lag_1","sales_lag_7","sales_lag_14","sales_lag_28","sales_lag_56",
    "promo_lag_1","promo_lag_7","promo_lag_28",
    "is_payday","holiday_weekday","holiday_weekend",
    "oil_above_mean","is_earthquake_window","dom_bucket_5"
]
simple_cols += [c for c in train_model.columns if c.startswith("dow_")]
simple_cols = [c for c in simple_cols if c in train_model.columns]

In [63]:
X_va = train_va[simple_cols].fillna(0)

max_per_col = X_va.max().sort_values(ascending=False).head(20)
max_per_col

sales_lag_56            24744.0
sales_lag_28            21180.0
sales_lag_1             18340.0
sales_lag_14            18340.0
sales_lag_7             18340.0
promo_lag_28              591.0
promo_lag_1               519.0
promo_lag_7               519.0
dom_bucket_5                6.0
dow_0                       1.0
dow_5                       1.0
dow_4                       1.0
dow_2                       1.0
dow_1                       1.0
holiday_weekend             1.0
holiday_weekday             1.0
is_payday                   1.0
dow_6                       1.0
is_earthquake_window        0.0
oil_above_mean              0.0
dtype: float64

In [64]:
X_va.describe().T.sort_values("max", ascending=False).head(20)[["min","mean","max"]]

,min,mean,max
sales_lag_56,0.0,493.217058,24744.0
sales_lag_28,0.0,477.061445,21180.0
sales_lag_1,0.0,469.062437,18340.0
sales_lag_14,0.0,475.761454,18340.0
sales_lag_7,0.0,477.100243,18340.0
promo_lag_28,0.0,8.144180,591.0
promo_lag_1,0.0,6.338203,519.0
promo_lag_7,0.0,6.648609,519.0
dom_bucket_5,0.0,2.571429,6.0
dow_0,0.0,0.142857,1.0


In [65]:
pred_va = 0.7*train_va["sales_lag_7"] + 0.3*train_va["sales_lag_14"]
mask = pred_va.notna()
print("Weighted lag RMSLE:", rmsle(train_va.loc[mask,"sales"], pred_va.loc[mask]))

Weighted lag RMSLE: 0.49855145243814497


In [66]:
pred_va2 = 0.5*train_va["sales_lag_1"] + 0.5*train_va["sales_lag_7"]
mask = pred_va2.notna()
print("Lag1+Lag7 RMSLE:", rmsle(train_va.loc[mask,"sales"], pred_va2.loc[mask]))

Lag1+Lag7 RMSLE: 0.4610427719606152


In [67]:
feature_cols = simple_cols  # simple

X_tr = train_tr[feature_cols].fillna(0)
y_tr = train_tr["y_log"]

X_va = train_va[feature_cols].fillna(0)
y_va = train_va["sales"].values

model = lgb.LGBMRegressor(
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_tr, y_tr)

pred = np.expm1(model.predict(X_va))
pred = np.maximum(pred, 0)

print("LightGBM RMSLE:", rmsle(y_va, pred))
print("Baseline lag1+lag7 RMSLE:", rmsle(y_va, (0.5*train_va["sales_lag_1"] + 0.5*train_va["sales_lag_7"]).values))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.041812 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1910
[LightGBM] [Info] Number of data points in the train set: 2851200, number of used features: 20
[LightGBM] [Info] Start training from score 2.945020
LightGBM RMSLE: 0.4119639060245271
Baseline lag1+lag7 RMSLE: 0.4610427719606152


In [68]:
ws = np.linspace(0, 1, 21)
best = (None, 1e9)

for w in ws:
    pred = w*train_va["sales_lag_1"] + (1-w)*train_va["sales_lag_7"]
    m = pred.notna()
    score = rmsle(train_va.loc[m,"sales"], pred.loc[m])
    if score < best[1]:
        best = (w, score)
best

(np.float64(0.45), np.float64(0.46058508573925044))

In [69]:
w = 0.45
pred_opt = w*train_va["sales_lag_1"] + (1-w)*train_va["sales_lag_7"]
m = pred_opt.notna()
print("Optimal weighted baseline RMSLE:", rmsle(train_va.loc[m,"sales"], pred_opt.loc[m]))

Optimal weighted baseline RMSLE: 0.46058508573925044


In [70]:
# train final
X_full = train_model[feature_cols].fillna(0)
y_full = train_model["y_log"]

model.fit(X_full, y_full)

# predict test
X_test = test_fe[feature_cols].fillna(0)
test_pred = np.expm1(model.predict(X_test))
test_pred = np.maximum(test_pred, 0)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.055615 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1914
[LightGBM] [Info] Number of data points in the train set: 2901096, number of used features: 20
[LightGBM] [Info] Start training from score 2.956567


In [71]:
sample = pd.read_csv("/Users/JulioJerez/Desktop/Docs. Personales/Repositorio/PROYECTOS/store-sales-forecasting/data/raw/store-sales-time-series-forecasting/sample_submission.csv")

sub = sample.copy()
sub["sales"] = test_pred

# sanity checks
print(sub.shape)
print(sub.head())

sub.to_csv("/Users/JulioJerez/Desktop/Docs. Personales/Repositorio/PROYECTOS/store-sales-forecasting/data/processed/submission_lgbm.csv", index=False)

(28512, 2)
        id     sales
0  3000888  4.068202
1  3000889  2.196262
2  3000890  2.115397
3  3000891  4.112711
4  3000892  1.319283


In [72]:
print("Any NaN?", sub["sales"].isna().any())
print("Any negative?", (sub["sales"] < 0).any())
print("min/median/max:", sub["sales"].min(), sub["sales"].median(), sub["sales"].max())

Any NaN? False
Any negative? False
min/median/max: 0.0 1.472149025182326 11728.440730659551


In [73]:
imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
imp.head(15)

sales_lag_1        7271
sales_lag_7        6988
sales_lag_56       6495
sales_lag_28       5570
sales_lag_14       5515
dom_bucket_5       3676
promo_lag_1        2627
promo_lag_7        2328
promo_lag_28       2158
holiday_weekday    1255
dow_6              1091
dow_0               998
oil_above_mean      954
dow_5               709
dow_4               663
dtype: int32

In [74]:
if "dcoilwtico" in train_model.columns and "dcoilwtico" not in feature_cols:
    feature_cols = feature_cols + ["dcoilwtico"]

In [75]:
if "promo_lag_14" in train_model.columns and "promo_lag_14" not in feature_cols:
    feature_cols = feature_cols + ["promo_lag_14"]

In [76]:
extra = [f"sales_lag_{i}" for i in range(2,7)]
extra = [c for c in extra if c in train_model.columns and c not in feature_cols]
feature_cols2 = feature_cols + extra

In [81]:
cat_cols = ["store_nbr", "family", "cluster"]
feature_cols_cat = feature_cols + cat_cols

# Asegurar dtype category
for c in cat_cols:
    train_tr[c] = train_tr[c].astype("category")
    train_va[c] = train_va[c].astype("category")

# Construir X sin fillna global
X_tr = train_tr[feature_cols_cat].copy()
X_va = train_va[feature_cols_cat].copy()

# Fillna SOLO en columnas numéricas
num_cols = [c for c in feature_cols_cat if c not in cat_cols]
X_tr[num_cols] = X_tr[num_cols].fillna(0)
X_va[num_cols] = X_va[num_cols].fillna(0)

y_tr = train_tr["y_log"]
y_va = train_va["sales"].values

In [82]:
model = lgb.LGBMRegressor(
    n_estimators=5000,
    learning_rate=0.05,
    num_leaves=128,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    force_row_wise=True
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_va, train_va["y_log"])],
    eval_metric="rmse",
    categorical_feature=cat_cols,
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(50)]
)

pred = np.expm1(model.predict(X_va))
pred = np.maximum(pred, 0)
print("LightGBM + categorical RMSLE:", rmsle(y_va, pred))

[LightGBM] [Info] Total Bins 2480
[LightGBM] [Info] Number of data points in the train set: 2851200, number of used features: 25
[LightGBM] [Info] Start training from score 2.945020
Training until validation scores don't improve for 100 rounds
[50]	valid_0's rmse: 0.455623	valid_0's l2: 0.207593
[100]	valid_0's rmse: 0.405187	valid_0's l2: 0.164177
[150]	valid_0's rmse: 0.402804	valid_0's l2: 0.162251
[200]	valid_0's rmse: 0.40304	valid_0's l2: 0.162441
[250]	valid_0's rmse: 0.402562	valid_0's l2: 0.162056
[300]	valid_0's rmse: 0.402111	valid_0's l2: 0.161693
[350]	valid_0's rmse: 0.401759	valid_0's l2: 0.16141
[400]	valid_0's rmse: 0.401687	valid_0's l2: 0.161353
[450]	valid_0's rmse: 0.401862	valid_0's l2: 0.161493
Early stopping, best iteration is:
[372]	valid_0's rmse: 0.401539	valid_0's l2: 0.161234
LightGBM + categorical RMSLE: 0.401525712521276


In [83]:
#  full train
cat_cols = ["store_nbr", "family", "cluster"]
feature_cols_cat = feature_cols + cat_cols

for c in cat_cols:
    train_model[c] = train_model[c].astype("category")
    test_fe[c] = test_fe[c].astype("category")

X_full = train_model[feature_cols_cat].copy()
X_test = test_fe[feature_cols_cat].copy()

num_cols = [c for c in feature_cols_cat if c not in cat_cols]
X_full[num_cols] = X_full[num_cols].fillna(0)
X_test[num_cols] = X_test[num_cols].fillna(0)

y_full = train_model["y_log"]

# re-fit model using best_iteration_
final_model = lgb.LGBMRegressor(
    n_estimators=model.best_iteration_,  # usa el best
    learning_rate=0.05,
    num_leaves=128,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    force_row_wise=True
)

final_model.fit(X_full, y_full, categorical_feature=cat_cols)

[LightGBM] [Info] Total Bins 2481
[LightGBM] [Info] Number of data points in the train set: 2901096, number of used features: 25
[LightGBM] [Info] Start training from score 2.956567


,boosting_type,'gbdt'
,num_leaves,128
,max_depth,-1
,learning_rate,0.05
,n_estimators,372
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [84]:
test_pred = np.expm1(final_model.predict(X_test))
test_pred = np.maximum(test_pred, 0)

sample = pd.read_csv("/Users/JulioJerez/Desktop/Docs. Personales/Repositorio/PROYECTOS/store-sales-forecasting/data/raw/store-sales-time-series-forecasting/sample_submission.csv")
sub = sample.copy()
sub["sales"] = test_pred

sub.to_csv("/Users/JulioJerez/Desktop/Docs. Personales/Repositorio/PROYECTOS/store-sales-forecasting/data/processed/submission_lgbm_cat.csv", index=False)
sub.head()

,id,sales
0,3000888,3.796843
1,3000889,2.620514
2,3000890,2.628892
3,3000891,4.210344
4,3000892,1.637162


In [85]:
imp = pd.Series(final_model.feature_importances_, index=feature_cols_cat).sort_values(ascending=False)
imp.head(20)

store_nbr          7145
family             6688
dcoilwtico         5298
sales_lag_1        4222
sales_lag_7        3877
sales_lag_56       2970
sales_lag_28       2804
sales_lag_14       2685
dom_bucket_5       2491
promo_lag_1        1069
holiday_weekday    1008
cluster             832
dow_6               731
promo_lag_28        731
promo_lag_7         726
dow_0               724
dow_5               609
promo_lag_14        609
dow_4               608
dow_2               353
dtype: int32